# NER gia phả — kiểm chéo 5 lượt trên gold người soát (25 cây)
**Tốn tiền (GPU Colab).** Chỉ train trên nhãn do người soát; mỗi cây làm test đúng một lần. Cần trong `MyDrive/family_tree_ner/`: `train_ner.py`, `vi_cv5.zip`.

In [ ]:
MODEL = "vinai/phobert-base"   # KHÔNG dùng phobert-base-v2 (AGPL)
EPOCHS, LR, BATCH = 20, 5e-5, 16
BASE = "/content/drive/MyDrive/family_tree_ner"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "transformers>=4.40"
import torch; assert torch.cuda.is_available(), "Chưa bật GPU"
print(torch.cuda.get_device_name(0))
!cd /content && unzip -qo {BASE}/vi_cv5.zip && ls vi_cv5

In [ ]:
import subprocess, json, shutil, os
os.makedirs(f"{BASE}/cv_results", exist_ok=True)
for k in range(5):
    out = f"/content/out/fold{k}"
    subprocess.run(["python", f"{BASE}/train_ner.py", "--model", MODEL, "--data-dir", f"/content/vi_cv5/fold{k}",
                    "--output-dir", out, "--test-split", "test", "--epochs", str(EPOCHS), "--lr", str(LR),
                    "--batch-size", str(BATCH)], check=True)
    shutil.copy(f"{out}/model_card.json", f"{BASE}/cv_results/fold{k}_model_card.json")
    shutil.rmtree(out)   # không lưu trọng số từng lượt (tiết kiệm Drive)
    c = json.load(open(f"{BASE}/cv_results/fold{k}_model_card.json"))
    print(f"== fold{k}: best_epoch={c['best_epoch']} dev_F1={c['dev']['micro']['f1']} test_F1={c['test']['micro']['f1']}")

In [ ]:
# Gộp 5 lượt: cộng tp/fp/fn rồi tính F1 (micro) — mỗi cây làm test đúng 1 lần
from collections import Counter
tot = {"micro": Counter(), "labels": {}}
for k in range(5):
    c = json.load(open(f"{BASE}/cv_results/fold{k}_model_card.json"))["test"]
    for key in ("tp","fp","fn"): tot["micro"][key] += c["micro"][key]
    for l, v in c["per_label"].items():
        d = tot["labels"].setdefault(l, Counter())
        for key in ("tp","fp","fn"): d[key] += v[key]
def prf(d):
    p = d["tp"]/(d["tp"]+d["fp"]) if d["tp"]+d["fp"] else 0; r = d["tp"]/(d["tp"]+d["fn"]) if d["tp"]+d["fn"] else 0
    return dict(P=round(p,3), R=round(r,3), F1=round(2*p*r/(p+r),3) if p+r else 0, **d)
print("GỘP 25 CÂY:", prf(tot["micro"]))
for l, d in tot["labels"].items(): print(l, prf(d))
json.dump({"pooled": prf(tot["micro"]), "per_label": {l: prf(d) for l, d in tot["labels"].items()}}, open(f"{BASE}/cv_results/pooled.json","w"), indent=1)

In [ ]:
from google.colab import runtime
runtime.unassign()   # NGẮT RUNTIME để dừng tính tiền